# LLM Inference Benchmark — Dataset & Traffic Feature Test

Tests the new dataset loaders, traffic simulation, and `benchmark_serving.py` CLI
added to `llm_benchmark`.

| Test | Feature | Key parameter(s) |
|------|---------|------------------|
| Smoke | Dataset loading | No server needed — just load & inspect each dataset |
| 1 | Dataset comparison | `random`, `sonnet`, `sharegpt`, `dolly` on baseline vLLM |
| 2 | Prefix caching | `sonnet` with `prefix_len=0` vs `prefix_len=400` |
| 3 | Fixed output length | `min_tokens=max_tokens` eliminates output token variability |
| 4 | Rate-controlled traffic | `request_rate=5.0` (Poisson) vs unlimited burst |
| CLI | `benchmark_serving.py` | CLI connecting to the running server |

A **single vLLM baseline server** is started in §4 and shared by all server-side tests.
Re-run the **Start Server** cell if it crashes between tests.

---
## 1. Install Dependencies

In [ ]:
!pip uninstall -y protobuf -q
!pip install "protobuf>=4.25,<5.0" -q
!pip install "vllm>=0.6.3" "openai>=1.51.0" "transformers>=4.44.2" \
    "tokenizers>=0.19.1" "accelerate" "GPUtil" "httpx>=0.27" "pandas" \
    "psutil" "torch" "gpustat" "nvidia-ml-py" "seaborn" "matplotlib" \
    "huggingface_hub" "datasets" "aiohttp" "nest_asyncio" -q

## 2. Hugging Face Authentication

In [ ]:
import os

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## 3. Download Model

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Downloading {MODEL_ID}...")
snapshot_download(MODEL_ID)
print("Done.")

## 4. Check Module Files

Upload the `llm_benchmark/` folder if running on Colab.

In [ ]:
import os

required = [
    "llm_benchmark/__init__.py",
    "llm_benchmark/benchmark_core.py",
    "llm_benchmark/server.py",
    "llm_benchmark/workloads.py",
    "llm_benchmark/metrics.py",
    "llm_benchmark/stats.py",
    "llm_benchmark/runner.py",
    "llm_benchmark/datasets.py",          # NEW
    "llm_benchmark/benchmark_serving.py", # NEW
]

missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("Missing files (upload llm_benchmark/ folder):")
    for f in missing:
        print(f"  {f}")
else:
    print("All required files present.")

---
## 5. Dataset Smoke Tests (No Server Needed)

Load a small sample from each dataset and print token-length statistics.
This validates the loaders work before touching any server.

In [ ]:
import numpy as np
from transformers import AutoTokenizer
from llm_benchmark.datasets import load_dataset_prompts, PromptSample

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

SMOKE_N = 30  # small sample for speed

datasets_to_test = [
    # (name,     kwargs)
    ("random",   {"input_len": 512, "output_len": 128}),
    ("sonnet",   {"input_len": 550, "output_len": 150}),
    ("dolly",    {"input_len": 512, "output_len": 128}),
    ("sharegpt", {"input_len": 512, "output_len": 128}),  # downloads HF dataset
]

print(f"{'Dataset':<12} {'N':>4} {'in_mean':>8} {'in_p50':>8} {'in_p95':>8} {'out_mean':>9} {'sample_prompt (first 80 chars)'}")
print("-" * 110)

smoke_results = {}
for name, kwargs in datasets_to_test:
    try:
        samples = load_dataset_prompts(
            dataset_name=name,
            num_prompts=SMOKE_N,
            tokenizer=tokenizer,
            **kwargs,
        )
        in_toks  = [s.input_tokens  for s in samples]
        out_toks = [s.output_tokens for s in samples]
        preview  = samples[0].prompt[:80].replace("\n", " ")
        print(
            f"{name:<12} {len(samples):>4} "
            f"{np.mean(in_toks):>8.0f} {np.percentile(in_toks,50):>8.0f} "
            f"{np.percentile(in_toks,95):>8.0f} {np.mean(out_toks):>9.0f}  "
            f"{preview!r}"
        )
        smoke_results[name] = samples
    except Exception as e:
        print(f"{name:<12}  ERROR: {e}")

print("\nSmoke tests complete.")

---
## 6. Start vLLM Baseline Server

A single server is shared by all subsequent tests.  
Re-run this cell if the server crashes between tests.

In [ ]:
import asyncio
import nest_asyncio
import pandas as pd
import numpy as np

nest_asyncio.apply()

from transformers import AutoTokenizer
from llm_benchmark import (
    detect_gpu, pick_dtype,
    ServerConfig, ServerProcess,
    run_workload,
)
from llm_benchmark.runner import run_workload_async
from llm_benchmark.datasets import load_dataset_prompts
from llm_benchmark.metrics import aggregate_metrics

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

gpu_info = detect_gpu()
dtype    = pick_dtype(gpu_info)

server_cfg = ServerConfig(
    engine="vllm",
    model_id=MODEL_ID,
    port=8000,
    dtype=dtype,
    gpu_mem_util=0.90,
    max_model_len=8192,
)

server = ServerProcess(server_cfg)
server.start()

BASE_URL = server_cfg.base_url()
print(f"Server running at {BASE_URL}")
print(f"GPU: {gpu_info['name']}  dtype: {dtype}")

# Warmup
print("Warming up...")
_ = run_workload(
    base_url=BASE_URL, model=MODEL_ID,
    prompts=["Hello, world!"] * 3,
    tokenizer=tokenizer, concurrency=3, max_tokens=32,
)
print("Warmup complete.")

### Helper: `bench_dataset()`

Loads a dataset, sends all prompts concurrently, and returns a summary dict.

In [ ]:
def bench_dataset(
    label,
    dataset_name,
    num_prompts=64,
    concurrency=32,
    max_tokens=128,
    input_len=512,
    output_len=128,
    prefix_len=0,
    request_rate=float("inf"),
    min_tokens=None,
    temperature=0.2,
):
    """
    Load *dataset_name*, send *num_prompts* to the running server,
    and return a one-row dict of results.
    """
    samples = load_dataset_prompts(
        dataset_name=dataset_name,
        num_prompts=num_prompts,
        tokenizer=tokenizer,
        input_len=input_len,
        output_len=output_len,
        prefix_len=prefix_len,
    )
    prompts = [s.prompt for s in samples]

    bm = asyncio.run(run_workload_async(
        base_url=BASE_URL,
        model=MODEL_ID,
        prompts=prompts,
        tokenizer=tokenizer,
        concurrency=concurrency,
        max_tokens=max_tokens,
        temperature=temperature,
        request_rate=request_rate,
        min_tokens=min_tokens,
    ))

    avg_in  = np.mean([s.input_tokens  for s in samples])
    avg_out = np.mean([s.output_tokens for s in samples])

    result = {
        "label":          label,
        "dataset":        dataset_name,
        "num_prompts":    num_prompts,
        "concurrency":    concurrency,
        "request_rate":   request_rate if request_rate < float("inf") else "inf",
        "min_tokens":     min_tokens,
        "avg_in_tok":     round(avg_in, 1),
        "avg_out_tok":    round(avg_out, 1),
        "success":        bm.successful_requests,
        "failed":         bm.failed_requests,
        "wall_s":         round(bm.wall_time_s, 2),
        "throughput_tok_s": bm.throughput_output_tok_s,
        "ttft_p50_ms":    bm.ttft_p50_ms,
        "ttft_p95_ms":    bm.ttft_p95_ms,
        "ttft_p99_ms":    bm.ttft_p99_ms,
        "itl_p50_ms":     bm.itl_p50_ms,
        "itl_p99_ms":     bm.itl_p99_ms,
        "req_lat_p99_ms": bm.request_latency_p99_ms,
    }

    sr = bm.successful_requests / max(bm.total_requests, 1) * 100
    print(
        f"  [{label}]  {bm.throughput_output_tok_s:>8.1f} tok/s  "
        f"TTFT p50/p99 {bm.ttft_p50_ms:>7.1f}/{bm.ttft_p99_ms:>7.1f} ms  "
        f"Success {sr:.0f}%"
    )
    return result


all_results = []  # accumulated across all tests
print("Helper ready.")

---
## Test 1 — Dataset Comparison

Run each of the four datasets through the baseline server with identical
concurrency and `max_tokens`.  Differences in throughput and TTFT reveal
how prompt-length distribution affects inference.

| Dataset | Prompts | Typical input length |
|---------|---------|----------------------|
| `random` | Synthetic | Exact 512 tokens |
| `sonnet` | Shakespeare | ~550 tokens |
| `dolly` | Databricks Dolly-15k | Variable 50-1500 tokens |
| `sharegpt` | Real ChatGPT convos | Variable, ±50% of 512 filter |

In [ ]:
print("=" * 70)
print("TEST 1: Dataset comparison (same concurrency, max_tokens=128)")
print("=" * 70)

N = 64
CONC = 32

for ds in ["random", "sonnet", "dolly", "sharegpt"]:
    r = bench_dataset(
        label=f"t1_{ds}",
        dataset_name=ds,
        num_prompts=N,
        concurrency=CONC,
        max_tokens=128,
        input_len=512,
        output_len=128,
    )
    all_results.append(r)

df_t1 = pd.DataFrame([r for r in all_results if r["label"].startswith("t1_")])
df_t1[["dataset", "avg_in_tok", "avg_out_tok", "throughput_tok_s",
       "ttft_p50_ms", "ttft_p99_ms", "success", "failed"]]

---
## Test 2 — Sonnet with Shared Prefix (Prefix Caching)

The `sonnet` loader supports a `prefix_len` parameter: every prompt starts
with an identical prefix of that many tokens, followed by a unique suffix.
With vLLM's automatic prefix caching, the 2nd+ requests should find the
prefix already in the KV cache, reducing their effective prefill work.

**Expected**: lower TTFT when `prefix_len > 0`.

In [ ]:
print("=" * 70)
print("TEST 2: Sonnet prefix caching (prefix_len=0 vs prefix_len=400)")
print("=" * 70)

for prefix_len, label in [(0, "t2_no_prefix"), (400, "t2_prefix_400")]:
    r = bench_dataset(
        label=label,
        dataset_name="sonnet",
        num_prompts=64,
        concurrency=32,
        max_tokens=150,
        input_len=550,
        output_len=150,
        prefix_len=prefix_len,
    )
    all_results.append(r)

df_t2 = pd.DataFrame([r for r in all_results if r["label"].startswith("t2_")])
df_t2[["label", "avg_in_tok", "throughput_tok_s", "ttft_p50_ms", "ttft_p99_ms", "itl_p50_ms"]]

---
## Test 3 — Fixed Output Length (`min_tokens = max_tokens`)

By default, models stop generating as soon as they produce an EOS token,
leading to variable output lengths.  This makes throughput comparisons
noisy because different configs finish at different times.

Setting `min_tokens = max_tokens = N` forces the server to generate exactly
N tokens per request, giving clean apples-to-apples throughput numbers.

**Expected**: higher variance in output tokens when `min_tokens=None`;  
near-zero variance and deterministic wall-time when `min_tokens=max_tokens`.

In [ ]:
print("=" * 70)
print("TEST 3: Fixed vs variable output length")
print("=" * 70)

FIXED_LEN = 128

for min_tok, label in [(None, "t3_variable"), (FIXED_LEN, "t3_fixed")]:
    r = bench_dataset(
        label=label,
        dataset_name="random",
        num_prompts=64,
        concurrency=32,
        max_tokens=FIXED_LEN,
        input_len=512,
        output_len=FIXED_LEN,
        min_tokens=min_tok,
    )
    all_results.append(r)

# Also compare actual output token counts across the two runs
df_t3 = pd.DataFrame([r for r in all_results if r["label"].startswith("t3_")])
df_t3[["label", "min_tokens", "avg_out_tok", "throughput_tok_s",
       "ttft_p50_ms", "req_lat_p99_ms", "wall_s"]]

---
## Test 4 — Rate-Controlled Traffic (Poisson Arrivals)

The default behaviour sends all requests at once (unlimited burst).
Real production traffic follows a **Poisson process**: each new request
arrives after an exponentially distributed wait of `1/rate` seconds.

This test compares:
- **Unlimited** — all 64 requests dispatched immediately (current default)
- **5 req/s** — requests trickle in at Poisson-distributed intervals
- **2 req/s** — lighter load

**Expected**: TTFT improves at lower rates (less queuing); throughput drops.
The wall time for rate-controlled runs is ≥ `num_prompts / rate` seconds.

In [ ]:
print("=" * 70)
print("TEST 4: Rate-controlled traffic (Poisson arrivals)")
print("=" * 70)

N = 32  # fewer prompts so even 2 req/s finishes in ~16 s

for rate, label in [
    (float("inf"), "t4_unlimited"),
    (5.0,          "t4_rate_5"),
    (2.0,          "t4_rate_2"),
]:
    rate_str = "unlimited" if rate == float("inf") else f"{rate:.0f} req/s"
    print(f"  Rate: {rate_str}")
    r = bench_dataset(
        label=label,
        dataset_name="sonnet",
        num_prompts=N,
        concurrency=N,   # concurrency > effective in-flight for low rates
        max_tokens=128,
        input_len=550,
        output_len=128,
        request_rate=rate,
    )
    all_results.append(r)

df_t4 = pd.DataFrame([r for r in all_results if r["label"].startswith("t4_")])
df_t4[["label", "request_rate", "wall_s", "throughput_tok_s",
       "ttft_p50_ms", "ttft_p99_ms", "itl_p50_ms", "success"]]

---
## Test 5 — CLI: `benchmark_serving.py`

The CLI connects to the **already-running** server (no server management code needed).
Useful for scripted benchmarking in CI or from the terminal.

In [ ]:
# Show the CLI help text
!python -m llm_benchmark.benchmark_serving --help

In [ ]:
# Sonnet benchmark via CLI — fixed 128-token output, 50 prompts, concurrency 16
!python -m llm_benchmark.benchmark_serving \
    --base-url http://localhost:8000/v1 \
    --model meta-llama/Llama-3.2-3B-Instruct \
    --dataset-name sonnet \
    --num-prompts 50 \
    --max-concurrency 16 \
    --fixed-output-len 128 \
    --percentile-metrics \
    --output-json cli_sonnet_results.json

In [ ]:
# ShareGPT benchmark via CLI — Poisson 3 req/s, variable output length
!python -m llm_benchmark.benchmark_serving \
    --base-url http://localhost:8000/v1 \
    --model meta-llama/Llama-3.2-3B-Instruct \
    --dataset-name sharegpt \
    --num-prompts 30 \
    --max-concurrency 30 \
    --request-rate 3.0 \
    --percentile-metrics

In [ ]:
# Inspect the JSON output from the sonnet CLI run
import json

if os.path.exists("cli_sonnet_results.json"):
    with open("cli_sonnet_results.json") as f:
        d = json.load(f)

    print("Key metrics from cli_sonnet_results.json:")
    for k in ["total_requests", "successful_requests", "wall_time_s",
               "throughput_output_tok_s", "ttft_p50_ms", "ttft_p99_ms",
               "itl_p50_ms", "itl_p99_ms", "_config"]:
        print(f"  {k}: {d.get(k)}")
else:
    print("cli_sonnet_results.json not found — did the CLI cell run successfully?")

---
## Stop Server

Run this when all tests are done.

In [ ]:
server.stop()
print("Server stopped.")

---
## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
df_all = pd.DataFrame(all_results)
print(f"{len(df_all)} total benchmark rows collected")
df_all[["label", "dataset", "throughput_tok_s", "ttft_p50_ms", "ttft_p99_ms", "success"]]

In [ ]:
# --- Plot 1: Dataset comparison (Test 1) ---
df_p = df_all[df_all["label"].str.startswith("t1_")].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
PALETTE = sns.color_palette("Set2", n_colors=4)

# Throughput
axes[0].bar(df_p["dataset"], df_p["throughput_tok_s"], color=PALETTE, edgecolor="black", linewidth=0.5)
axes[0].set_title("Output Throughput", fontweight="bold")
axes[0].set_ylabel("tok/s")

# TTFT p50 vs p99
x = range(len(df_p))
w = 0.35
axes[1].bar([i - w/2 for i in x], df_p["ttft_p50_ms"], width=w, label="p50", color=PALETTE[0], edgecolor="black", linewidth=0.4)
axes[1].bar([i + w/2 for i in x], df_p["ttft_p99_ms"], width=w, label="p99", color=PALETTE[1], edgecolor="black", linewidth=0.4)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(df_p["dataset"].tolist())
axes[1].set_title("TTFT p50 vs p99", fontweight="bold")
axes[1].set_ylabel("ms (lower is better)")
axes[1].legend()

# Avg input token length
axes[2].bar(df_p["dataset"], df_p["avg_in_tok"], color=PALETTE, edgecolor="black", linewidth=0.5)
axes[2].set_title("Avg Input Tokens", fontweight="bold")
axes[2].set_ylabel("tokens")

fig.suptitle("Test 1: Dataset Comparison (same concurrency=32, max_tokens=128)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plot_dataset_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Plot 2: Prefix caching (Test 2) ---
df_p = df_all[df_all["label"].str.startswith("t2_")].copy()
labels = df_p["label"].str.replace("t2_", "").tolist()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].bar(labels, df_p["ttft_p50_ms"], color=sns.color_palette("Set2", 2), edgecolor="black", linewidth=0.5)
axes[0].set_title("TTFT p50 — Prefix vs No Prefix", fontweight="bold")
axes[0].set_ylabel("ms (lower is better)")

axes[1].bar(labels, df_p["throughput_tok_s"], color=sns.color_palette("Set2", 2), edgecolor="black", linewidth=0.5)
axes[1].set_title("Throughput — Prefix vs No Prefix", fontweight="bold")
axes[1].set_ylabel("tok/s (higher is better)")

fig.suptitle("Test 2: Sonnet Shared-Prefix Effect on Prefix Caching",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plot_prefix_caching.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Plot 3: Fixed vs variable output length (Test 3) ---
df_p = df_all[df_all["label"].str.startswith("t3_")].copy()
labels = df_p["label"].str.replace("t3_", "").tolist()

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
pal = sns.color_palette("Set2", 2)

axes[0].bar(labels, df_p["avg_out_tok"], color=pal, edgecolor="black", linewidth=0.5)
axes[0].set_title("Avg Output Tokens", fontweight="bold")
axes[0].set_ylabel("tokens")

axes[1].bar(labels, df_p["throughput_tok_s"], color=pal, edgecolor="black", linewidth=0.5)
axes[1].set_title("Throughput", fontweight="bold")
axes[1].set_ylabel("tok/s")

axes[2].bar(labels, df_p["wall_s"], color=pal, edgecolor="black", linewidth=0.5)
axes[2].set_title("Wall Time", fontweight="bold")
axes[2].set_ylabel("seconds")

fig.suptitle("Test 3: Fixed Output Length (min_tokens=max_tokens=128) vs Variable",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plot_fixed_output.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Plot 4: Rate-controlled traffic (Test 4) ---
df_p = df_all[df_all["label"].str.startswith("t4_")].copy()
labels = df_p["label"].str.replace("t4_", "").tolist()

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
pal = sns.color_palette("Set2", 3)

axes[0].bar(labels, df_p["throughput_tok_s"], color=pal, edgecolor="black", linewidth=0.5)
axes[0].set_title("Throughput", fontweight="bold")
axes[0].set_ylabel("tok/s")

x = range(len(df_p))
w = 0.35
axes[1].bar([i - w/2 for i in x], df_p["ttft_p50_ms"], width=w, label="p50",
            color=pal[0], edgecolor="black", linewidth=0.4)
axes[1].bar([i + w/2 for i in x], df_p["ttft_p99_ms"], width=w, label="p99",
            color=pal[1], edgecolor="black", linewidth=0.4)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels)
axes[1].set_title("TTFT p50 vs p99", fontweight="bold")
axes[1].set_ylabel("ms (lower is better)")
axes[1].legend()

axes[2].bar(labels, df_p["wall_s"], color=pal, edgecolor="black", linewidth=0.5)
axes[2].set_title("Total Wall Time", fontweight="bold")
axes[2].set_ylabel("seconds")

fig.suptitle("Test 4: Poisson Rate Control (unlimited vs 5 req/s vs 2 req/s)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plot_rate_control.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Full results table ---
print("=" * 80)
print("ALL RESULTS")
print("=" * 80)
display_cols = ["label", "dataset", "avg_in_tok", "avg_out_tok",
                "throughput_tok_s", "ttft_p50_ms", "ttft_p99_ms",
                "itl_p50_ms", "wall_s", "success", "failed"]
pd.set_option("display.float_format", "{:.1f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
print(df_all[display_cols].to_string(index=False))

df_all.to_csv("dataset_feature_results.csv", index=False)
print("\nSaved -> dataset_feature_results.csv")

---
## Export Results

In [ ]:
import glob

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

export_files = (
    ["dataset_feature_results.csv", "cli_sonnet_results.json"]
    + sorted(glob.glob("plot_*.png"))
)

for f in export_files:
    if os.path.exists(f):
        print(f"  {f}")
        if IN_COLAB:
            colab_files.download(f)
    else:
        print(f"  {f}  (not found)")